# Entity tagging
Tag English and Vietnamese articles with GLiNER and review mentions in pandas. Custom HTML rendering is replaced by an entity table and grouped counts.

In [1]:
from utils import setup_notebook, to_dataframe, write_json

from pathlib import Path

from IPython.display import display

from data_platform.digest.gliner import extract_entities

project_root = setup_notebook()


## 1. Configure and inspect inputs

In [5]:
input_paths = sorted((project_root / ".data").glob("*.md"))
model_id = "urchade/gliner_multi-v2.1"
labels = ["company name", "date", "email", "person name"]
threshold = 0.5
chunk_words = 200
chunk_overlap = 40
run_live = True
local_files_only = True
articles = {path.name: path.read_text(encoding="utf-8") for path in input_paths}
display(
    to_dataframe(
        [{"article": name, "characters": len(text)} for name, text in articles.items()],
        columns=["article", "characters"],
    )
)


,article,characters
0,article-en-01.md,6835
1,article-vi-01.md,4920


## 2. Tag and review
Offsets use original Python character positions. Duplicate and conflicting overlaps are resolved by confidence.

In [6]:
results = {}
if run_live:
    from gliner import GLiNER

    model = GLiNER.from_pretrained(model_id, local_files_only=local_files_only)
    model.to("cpu")
    model.eval()
    results = {
        name: extract_entities(
            text,
            model=model,
            labels=labels,
            threshold=threshold,
            chunk_words=chunk_words,
            chunk_overlap=chunk_overlap,
        )
        for name, text in articles.items()
    }

entities_df = to_dataframe(
    [
        {"article": name, **entity}
        for name, entities in results.items()
        for entity in entities
    ],
    columns=["article", "text", "label", "score", "start", "end"],
)
display(entities_df)
display(entities_df.groupby(["article", "label"]).size().rename("mentions").to_frame())


W1121 03:45:21.654000 35933 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Users/chuminhquan/projects/data-platform/.venv/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/Users/chuminhquan/projects/data-platform/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
[transformers] The tokenizer you are loading from 'microsoft/mdeberta-v3-base' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this 

,article,text,label,score,start,end
0,article-en-01.md,28-10-2026,date,0.526754,109,119
1,article-en-01.md,"July 31, 2027",date,0.681974,484,497
2,article-en-01.md,Mr. Nguyen Viet Quang,person name,0.682002,1373,1394
3,article-en-01.md,Mr. Dan Levy,person name,0.699341,1844,1856
4,article-en-01.md,Mr. Peter Loxton,person name,0.804546,2290,2306
5,article-en-01.md,Vingroup,company name,0.694791,3934,3942
6,article-en-01.md,Legends Global,company name,0.738746,3947,3961
7,article-en-01.md,VinFast,company name,0.656858,3963,3970
8,article-en-01.md,Mr. Paul Sergeant OBE,person name,0.789453,4518,4539
9,article-en-01.md,VINGROUP,company name,0.599911,4911,4919


mentions
article          label                 
article-en-01.md company name        11
                 date                 2
                 person name          4
article-vi-01.md company name        11

## 3. Filter mentions
Edit the label or minimum score to focus the review without rerunning inference.

In [4]:
selected_label = "company name"
minimum_score = 0.5
display(
    entities_df.loc[
        (entities_df["label"] == selected_label)
        & (entities_df["score"] >= minimum_score)
    ]
)


,article,text,label,score,start,end


## 4. Export (optional)
Saving replaces per-article CSV and JSON outputs. JSON includes the source text and model settings so offsets remain interpretable.

In [ ]:
save_output = False
if save_output and results:
    output_dir = project_root / ".data/gliner"
    output_dir.mkdir(parents=True, exist_ok=True)
    for name, entities in results.items():
        stem = Path(name).stem
        entities_df.loc[entities_df["article"] == name].drop(columns="article").to_csv(
            output_dir / f"{stem}.entities.csv", index=False
        )
        payload = {
            "source_file": name,
            "text": articles[name],
            "model": model_id,
            "labels": labels,
            "threshold": threshold,
            "chunk_words": chunk_words,
            "chunk_overlap": chunk_overlap,
            "entities": entities,
        }
        write_json(output_dir / f"{stem}.entities.json", payload)
